# E-Commerce Churn Prediction
EDA → Features → Model → Evaluation

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, warnings
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, RocCurveDisplay, ConfusionMatrixDisplay, confusion_matrix
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
import joblib
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)
print('Libraries loaded')

## 1. Load & Explore Data

In [ ]:
df = pd.read_csv('../data/raw/ecommerce_churn.csv')
print(f'Shape: {df.shape} | Churn rate: {df.churn.mean():.1%}')
df.head()

In [ ]:
df.describe().T

## 2. Target Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['churn'].value_counts().plot(kind='bar', ax=axes[0], color=['#4C72B0','#DD8452'], title='Churn Counts')
axes[1].pie(df['churn'].value_counts(), labels=['Retained','Churned'], autopct='%1.1f%%', colors=['#4C72B0','#DD8452'])
axes[1].set_title('Churn Split')
plt.tight_layout(); plt.show()

## 3. Feature Distributions

In [ ]:
cols = ['tenure_months','hours_on_app','satisfaction_score','day_since_last_order','cashback_amount','order_count_l6m']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flatten(), cols):
    df[df.churn==0][col].hist(ax=ax, alpha=0.6, bins=25, label='Retained', color='#4C72B0')
    df[df.churn==1][col].hist(ax=ax, alpha=0.6, bins=25, label='Churned',  color='#DD8452')
    ax.set_title(col); ax.legend()
plt.suptitle('Distributions by Churn', y=1.01); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
corr = df.select_dtypes(include=np.number).corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, mask=np.triu(np.ones_like(corr, dtype=bool)))
plt.title('Correlation Matrix'); plt.show()

## 4. Feature Engineering

In [ ]:
df_fe = df.copy()
df_fe['recency_engagement'] = df_fe['day_since_last_order'] / (df_fe['hours_on_app'] + 1)
df_fe['coupon_order_ratio'] = df_fe['coupon_used'] / (df_fe['order_count_l6m'] + 1)
df_fe['high_value_flag']    = (df_fe['cashback_amount'] > df_fe['cashback_amount'].median()).astype(int)
df_fe['multi_device_flag']  = (df_fe['devices_registered'] >= 3).astype(int)
print('Engineered features added')
df_fe[['recency_engagement','coupon_order_ratio','high_value_flag','multi_device_flag']].describe().T

## 5. Model Training

In [ ]:
num_feats = ['tenure_months','warehouse_to_home_km','hours_on_app','devices_registered',
             'order_count_l6m','order_amount_hike_pct','coupon_used','day_since_last_order',
             'cashback_amount','satisfaction_score','complain','number_of_address','city_tier',
             'recency_engagement','coupon_order_ratio','high_value_flag','multi_device_flag']
cat_feats = ['preferred_login_device','preferred_payment_mode','preferred_order_cat','gender']
X = df_fe.drop(columns=['customer_id','churn'])
y = df_fe['churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')

In [ ]:
num_pipe = Pipeline([('imp',SimpleImputer(strategy='median')),('scl',StandardScaler())])
cat_pipe = Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('ohe',OneHotEncoder(handle_unknown='ignore',sparse_output=False))])
preprocessor = ColumnTransformer([('num',num_pipe,num_feats),('cat',cat_pipe,cat_feats)])
model = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('rf', RandomForestClassifier(n_estimators=300, max_depth=12, class_weight='balanced', random_state=42, n_jobs=-1))
])
model.fit(X_train, y_train)
print('Training complete')

## 6. Evaluation

In [ ]:
y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:,1]
print(f'ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}')
print(classification_report(y_test, y_pred, target_names=['Retained','Churned']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[0])
axes[0].plot([0,1],[0,1],'k--'); axes[0].set_title('ROC Curve')
ConfusionMatrixDisplay(confusion_matrix(y_test,y_pred),display_labels=['Retained','Churned']).plot(ax=axes[1],cmap='Blues',colorbar=False)
axes[1].set_title('Confusion Matrix')
plt.tight_layout(); plt.show()

## 7. Feature Importances

In [ ]:
ohe_cats = model.named_steps['preprocessor'].named_transformers_['cat'].named_steps['ohe'].get_feature_names_out(cat_feats).tolist()
all_feats = num_feats + ohe_cats
fi_df = pd.DataFrame({'feature':all_feats,'importance':model.named_steps['rf'].feature_importances_})
fi_df = fi_df.sort_values('importance',ascending=False).head(15)
plt.figure(figsize=(10,6))
sns.barplot(data=fi_df, x='importance', y='feature', palette='Blues_d')
plt.title('Top 15 Features'); plt.tight_layout(); plt.show()

## 8. Risk Tiering & Business Impact

In [ ]:
risk = pd.cut(y_proba,[0,0.35,0.65,1.0],labels=['Low','Medium','High'])
summary = pd.DataFrame({'risk':risk,'churn':y_test.values})
print('Risk tier distribution:')
print(summary.risk.value_counts())
print('\nActual churn rate per tier:')
print(summary.groupby('risk')['churn'].mean().map('{:.1%}'.format))

## 9. Save Model

In [ ]:
import os; os.makedirs('../models/artifacts', exist_ok=True)
joblib.dump(model, '../models/artifacts/churn_model_notebook.joblib')
print('Model saved to ../models/artifacts/churn_model_notebook.joblib')